In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

In [0]:
races_schema= StructType([StructField("raceId", IntegerType(), True),
                          StructField("year", IntegerType(), True),
                          StructField("round", IntegerType(), True),
                          StructField("circuitId", IntegerType(), True),
                          StructField("name", StringType(), True),
                          StructField("date", DateType(), True),
                          StructField("time", StringType(), True),
                          StructField("url", StringType(), True)])

In [0]:
df_races= spark.read\
.schema(races_schema)\
.option("header", "true")\
.csv("abfss://formula1@formula1adlssai.dfs.core.windows.net/raw/races.csv")

In [0]:
display(df_races)

In [0]:
from pyspark.sql.functions import to_timestamp, concat, col,lit, current_timestamp

In [0]:
df_races_processed = df_races\
.withColumn(
    "races_timestamp",
    to_timestamp(concat(col("date"),lit(" "),col("time")), "yyyy-MM-dd HH:mm:ss")
)\
.withColumn("ingestion_date", current_timestamp())\
.drop("time")\
.drop("date")\
.drop("url")\
.withColumnRenamed("raceId", "race_id")\
.withColumnRenamed("circuitId", "circuit_id")\
.withColumnRenamed("year", "race_year")


In [0]:
display(df_races_processed)

In [0]:
df_races_processed.write.mode("overwrite").format("parquet").save("abfss://formula1@formula1adlssai.dfs.core.windows.net/processed/races/")

In [0]:
%fs ls "abfss://formula1@formula1adlssai.dfs.core.windows.net/processed/races"